In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm

In [3]:
!unzip -q /content/drive/MyDrive/cityscapes.zip -d /content/

In [ ]:
CITYSCAPES_ROOT = "/content/cityscapes"
BATCH_SIZE = 32
IMG_SIZE = (512, 1024)
NUM_CLASSES = 19
NUM_EPOCHS = 20
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SUBSET_SIZE = None # full size
VAL_SUBSET_SIZE = 500

print(f"Device: {DEVICE}")

Device: cuda


In [3]:
IGNORE_INDEX = 255

In [ ]:
# map raw labelIds from _gtFine_labelIds.png to 19-class trainIds
LABELID_TO_TRAINID = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7,
    21: 8, 22: 9, 23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18,
}
_LABELID_LUT = np.full(256, IGNORE_INDEX, dtype=np.uint8)
for _lid, _tid in LABELID_TO_TRAINID.items():
    _LABELID_LUT[_lid] = _tid

In [ ]:
class CityscapesDataset(Dataset):
    def __init__(self, root, split="train", img_size=(512, 1024), subset=None):
        self.root = root
        self.split = split
        self.img_size = img_size

        img_dir = os.path.join(root, "leftImg8bit", split)
        lbl_dir = os.path.join(root, "gtFine", split)

        self.images = []
        self.labels = []

        for city in sorted(os.listdir(img_dir)):
            city_dir = os.path.join(img_dir, city)
            if city.startswith(".") or not os.path.isdir(city_dir):
                continue
            for fname in sorted(os.listdir(city_dir)):
                if fname.endswith("_leftImg8bit.png"):
                    img_path = os.path.join(img_dir, city, fname)
                    lbl_path = os.path.join(lbl_dir, city, fname.replace("_leftImg8bit.png", "_gtFine_labelIds.png"))
                    self.images.append(img_path)
                    self.labels.append(lbl_path)

        if subset is not None:
            idx = np.random.choice(len(self.images), min(subset, len(self.images)), replace=False)
            self.images = [self.images[i] for i in idx]
            self.labels = [self.labels[i] for i in idx]

        self.img_transform = transforms.Compose([
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        print(f"  {split} dataset: {len(self.images)} images")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        lbl = Image.open(self.labels[idx])

        img = self.img_transform(img)

        lbl = lbl.resize((self.img_size[1], self.img_size[0]), resample=Image.NEAREST)
        lbl = _LABELID_LUT[np.array(lbl, dtype=np.uint8)]
        lbl = torch.from_numpy(lbl).long()

        return img, lbl

In [ ]:
class SegmentationHead(nn.Module):
    def __init__(self, in_channels, num_classes, scale_factor=32):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_classes, kernel_size=1),
        )
        self.scale_factor = scale_factor

    def forward(self, x):
        x = self.decoder(x)
        x = nn.functional.interpolate(x, scale_factor=self.scale_factor, mode="bilinear", align_corners=False)
        return x

In [ ]:
def get_backbone_and_head(model_name, freeze_backbone=False):
    if model_name in ("random", "supervised"):
        from torchvision.models import resnet50, ResNet50_Weights
        weights = ResNet50_Weights.IMAGENET1K_V1 if model_name == "supervised" else None
        backbone = resnet50(weights=weights)

        # remove the classification head and keep everything up to layer 4
        backbone = nn.Sequential(*list(backbone.children())[:-2])  # 2048 * H/32 * W/32
        in_channels = 2048
        scale_factor = 32

    elif model_name == "moco_v3":
        from torchvision.models import resnet50
        print("  Loading MoCo v3 ResNet-50 from official checkpoint...")
        backbone = resnet50(weights=None)

        ckpt_url = "https://dl.fbaipublicfiles.com/moco-v3/r-50-1000ep/r-50-1000ep.pth.tar"
        ckpt = torch.hub.load_state_dict_from_url(ckpt_url, map_location="cpu", check_hash=False)
        state = ckpt.get("state_dict", ckpt)

        new_state = {}
        for k, v in state.items():
            if k.startswith("module.base_encoder."):
                nk = k[len("module.base_encoder."):]
            elif k.startswith("base_encoder."):
                nk = k[len("base_encoder."):]
            else:
                continue
            if nk.startswith("fc.") or nk.startswith("head.") or "predictor" in nk:
                continue
            new_state[nk] = v

        missing, unexpected = backbone.load_state_dict(new_state, strict=False)
        print(f"    loaded MoCo v3 weights | missing={len(missing)} unexpected={len(unexpected)}")

        backbone = nn.Sequential(*list(backbone.children())[:-2])
        in_channels = 2048
        scale_factor = 32

    elif model_name == "dino":
        print("  Loading DINO ViT-S/16 from torch.hub...")
        backbone = torch.hub.load("facebookresearch/dino:main", "dino_vits16", pretrained=True)
        in_channels = 384  # ViT-S hidden dim
        scale_factor = 16

        # wrap backbone to output spatial features instead of CLS token
        backbone = DinoBackbone(backbone, patch_size=16)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    if freeze_backbone:
        for param in backbone.parameters():
            param.requires_grad = False
        print(f"  Backbone frozen (linear probe mode)")

    head = SegmentationHead(in_channels, NUM_CLASSES, scale_factor=scale_factor)
    return backbone, head

In [ ]:
class DinoBackbone(nn.Module):
    def __init__(self, vit, patch_size=16):
        super().__init__()
        self.vit = vit
        self.patch_size = patch_size

    def forward(self, x):
        B, C, H, W = x.shape
        # get all tokens (CLS + patches)
        features = self.vit.get_intermediate_layers(x, n=1)[0]  # B * (1 + num_patches) * dim
        features = features[:, 1:, :]  # remove CLS token - B * num_patches * dim

        # reshape to spatial grid
        h_patches = H // self.patch_size
        w_patches = W // self.patch_size
        features = features.reshape(B, h_patches, w_patches, -1)
        features = features.permute(0, 3, 1, 2)  # B * dim * h_patches * w_patches
        return features

In [9]:
class SegmentationModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)